# RTM3D/KM3D ResNet-18 — Core ML head export

**Revision:** `RTM3D-KM3D-COREML-EXPORT-2026-10-07-r3`

Run this after the fixed-16 r7 smoke passed. It safely reloads the same pinned checkpoint, converts the fixed 1280×384 network heads to an FP32 Core ML ML Program, and packages 16 evenly spaced Chen validation images with their PyTorch reference heads. The native Mac audit checks all fixtures with the unchanged max-absolute tolerance of 2e-4 and reports per-head and per-image errors. This notebook does not train, evaluate AP, port the calibration-dependent 3D decoder, or qualify an iPhone build.

After the ZIP downloads, extract it on your Mac and run `scripts/audit_rtm3d_km3d_res18_coreml.py` for native Core ML parity. **Do not connect/use the iPhone yet**: first pass Mac parity, then validate 3D geometry decoding, then integrate and profile the iPhone app. The package by itself returns raw heads, not 3D boxes.


In [ ]:
from pathlib import Path
from datetime import datetime
import json, subprocess, sys
from google.colab import drive, files
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/mobile_adas3d')
PROJECT_URL = 'https://github.com/Ali-RT/mobile_adas3d.git'
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', 'main', '--single-branch', PROJECT_URL, str(PROJECT_DIR)], check=True)
origin = subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'remote', 'get-url', 'origin'], text=True).strip()
if origin.startswith('git@github.com:'): origin = 'https://github.com/' + origin.split(':', 1)[1]
if origin.rstrip('/').removesuffix('.git').lower() != 'https://github.com/ali-rt/mobile_adas3d':
    raise RuntimeError(f'Unexpected project origin: {origin!r}')
if not (PROJECT_DIR / '.git').exists(): raise RuntimeError(f'{PROJECT_DIR} is not a Git checkout; preserve it and choose a new path.')
branch = subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'branch', '--show-current'], text=True).strip()
if branch != 'main': raise RuntimeError(f'Expected main, found {branch!r}; preserve checkout and use a fresh path.')
dirty = subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'status', '--porcelain', '--untracked-files=no'], text=True).strip()
if dirty: raise RuntimeError('MobileADAS3D has tracked edits; preserve them and use a clean checkout.')
subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', 'main'], check=True)
SCRIPT = PROJECT_DIR / 'scripts/export_rtm3d_km3d_res18_coreml.py'
if not SCRIPT.is_file(): raise FileNotFoundError(f'{SCRIPT} missing after updating main; confirm export revision is published.')
SPLIT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
OUTPUT_ROOT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/rtm3d_km3d_res18_r1')
RTM3D_REPO = Path('/content/RTM3D_KM3D_R1_20261007')
EXPECTED_COMMIT = '888c379e79d8a6d134f06a9b7d669118679e06dc'
RTM3D_URL = 'https://github.com/Banconxuan/RTM3D.git'
CHECKPOINT_ID = '14ww6mxtitO9aDszZN3ai8N7U1doehvi8'
if not (SPLIT_DIR / 'val.txt').is_file(): raise FileNotFoundError(SPLIT_DIR / 'val.txt')
import torch
print('MobileADAS3D main:', subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('Output root:', OUTPUT_ROOT)


## Prepare the exact source, trained checkpoint, and Chen validation images

This cell preserves dirty RTM3D folders and selects a clean pinned checkout. The fixture uses 16 evenly spaced IDs from the same 3,769-image Chen validation split and precisely the same preprocessing as the passed smoke.

In [ ]:
import subprocess
def inspect_repo(path):
    if not path.exists(): return False, 'missing'
    try:
        head = subprocess.check_output(['git','-C',str(path),'rev-parse','HEAD'],text=True,stderr=subprocess.DEVNULL).strip()
        origin = subprocess.check_output(['git','-C',str(path),'remote','get-url','origin'],text=True,stderr=subprocess.DEVNULL).strip()
        dirty = subprocess.check_output(['git','-C',str(path),'status','--porcelain','--untracked-files=all'],text=True).strip()
    except subprocess.CalledProcessError: return False, 'not a Git checkout'
    norm = origin.rstrip('/').removesuffix('.git').lower()
    issues = ([f'commit {head}' ] if head != EXPECTED_COMMIT else []) + (['unexpected origin'] if norm != RTM3D_URL.removesuffix('.git').lower() else []) + (['modified or untracked files'] if dirty else [])
    return not issues, '; '.join(issues) if issues else 'clean pinned checkout'
ok, why = inspect_repo(RTM3D_REPO)
if RTM3D_REPO.exists() and not ok:
    original = RTM3D_REPO; suffix = 1
    while True:
        candidate = original.with_name(f'{original.name}_fresh{suffix}')
        valid, _ = inspect_repo(candidate)
        if not candidate.exists() or valid: RTM3D_REPO = candidate; break
        suffix += 1
    print(f'Preserving existing RTM3D checkout ({why}); selected {RTM3D_REPO}')
if not RTM3D_REPO.exists():
    subprocess.run(['git','clone',RTM3D_URL,str(RTM3D_REPO)],check=True)
    subprocess.run(['git','-C',str(RTM3D_REPO),'checkout','--detach',EXPECTED_COMMIT],check=True)
ok, why = inspect_repo(RTM3D_REPO)
if not ok: raise RuntimeError(f'RTM3D source verification failed: {why}')
VAL_IDS = [line.strip() for line in (SPLIT_DIR/'val.txt').read_text().splitlines() if line.strip()]
if len(VAL_IDS) != 3769 or len(set(VAL_IDS)) != 3769: raise RuntimeError(f'Expected 3769 unique Chen val IDs, got {len(VAL_IDS)}')
DATA_CANDIDATES = [Path('/content/kitti_m67'),Path('/content/kitti_m66'),Path('/content/kitti_m65'),Path('/content/kitti_m64'),Path('/content/kitti_m62'),Path('/content/kitti_m61'),Path('/content/kitti'),Path('/content/monodetr_kitti_a2'),Path('/content/drive/MyDrive/datasets/kitti')]
DATASET_ROOT = next((root for root in DATA_CANDIDATES if (root/'training/image_2'/f'{VAL_IDS[0]}.png').is_file() and (root/'training/calib'/f'{VAL_IDS[0]}.txt').is_file()),None)
if DATASET_ROOT is None: raise FileNotFoundError('Could not find KITTI val image and calibration. Add your root to DATA_CANDIDATES.')
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
CHECKPOINT = OUTPUT_ROOT/'model_res18_1.pth'
if not CHECKPOINT.is_file():
    subprocess.run([sys.executable,'-m','pip','install','-q','gdown','coremltools==9.0','opencv-python-headless'],check=True)
    subprocess.run([sys.executable,'-m','gdown',f'https://drive.google.com/uc?id={CHECKPOINT_ID}','-O',str(CHECKPOINT)],check=True)
else:
    subprocess.run([sys.executable,'-m','pip','install','-q','coremltools==9.0','opencv-python-headless'],check=True)
if CHECKPOINT.stat().st_size < 1_000_000: raise RuntimeError('Checkpoint file looks incomplete')
print('Source:', subprocess.check_output(['git','-C',str(RTM3D_REPO),'rev-parse','HEAD'],text=True).strip())
print('Checkpoint:', CHECKPOINT, CHECKPOINT.stat().st_size, 'bytes')
print('Representative image:', DATASET_ROOT/'training/image_2'/f'{VAL_IDS[0]}.png')
print('All inputs ready. No phone connection needed.')


## Export the FP32 network heads and create the multi-image Mac parity bundle

This verifies checkpoint hashes and strict loading, TorchScript parity, finite raw heads, fixed output shapes, and absence of Core ML custom ops. It records PyTorch references for all 16 fixtures. It deliberately does not load/predict the Core ML package in Colab; that happens on macOS in the next gate.

In [ ]:
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
EXPORT_DIR = OUTPUT_ROOT / f'coreml_export_r3_{RUN_TAG}'
LOG_DIR = OUTPUT_ROOT / 'colab_logs'; LOG_DIR.mkdir(parents=True,exist_ok=True)
EXPORT_LOG = LOG_DIR / f'coreml_export_r3_{RUN_TAG}.log'
command = [sys.executable,'-u',str(SCRIPT),'--repo',str(RTM3D_REPO),'--checkpoint',str(CHECKPOINT),'--dataset-root',str(DATASET_ROOT),'--split-dir',str(SPLIT_DIR),'--output-dir',str(EXPORT_DIR)]
print('+',' '.join(command),flush=True)
with EXPORT_LOG.open('w') as log:
    process = subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:
        print(line,end='',flush=True); log.write(line); log.flush()
    return_code = process.wait()
if return_code: raise RuntimeError(f'RTM3D Core ML export failed (exit {return_code}); full combined log: {EXPORT_LOG}')
EXPORT_REPORT = json.loads((EXPORT_DIR/'rtm3d_coreml_export.json').read_text())
assert EXPORT_REPORT['complete'] and EXPORT_REPORT['trace_parity_passed'] and not EXPORT_REPORT['mil_has_custom_op']
assert not EXPORT_REPORT['geometry_decode_included'] and not EXPORT_REPORT['iphone_performance_measured']
BUNDLE = EXPORT_DIR/'rtm3d_km3d_res18_coreml_export.zip'
print('Export report:', EXPORT_DIR/'rtm3d_coreml_export.json')
print('Durable log:', EXPORT_LOG)
print('Download this bundle, then move it to your Mac for native Core ML parity:', BUNDLE)
files.download(str(BUNDLE))


## After export: run native Core ML parity on the Mac

Send the downloaded `rtm3d_km3d_res18_coreml_export.zip` back in this project chat for the native Mac audit. It runs CPU-only Core ML predictions on all 16 fixtures, keeps the existing `2e-4` max-absolute gate, and writes per-image/per-head diagnostics. Do not proceed to geometry decoding or iPhone integration unless the report says `complete: true`.